# Scraping de un artículo de Alkomprar con Selenium
Práctica de Análisis y herramientas de extracción de datos.

**Qué hace este cuaderno:** abre la página de un producto de Alkomprar, saca el nombre, el precio, el precio anterior, la calificación y el número de opiniones, calcula el descuento y guarda todo en un archivo CSV.

Ejecuta las celdas en orden. Solo tienes que cambiar el enlace del producto en la celda 3.

## 1. Instalar lo necesario
Se instala Selenium y Google Chrome (Colab no trae navegador con pantalla, así que se usa en modo *headless*).

In [3]:
!pip install -q selenium pandas
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -qq ./google-chrome-stable_current_amd64.deb

Selecting previously unselected package at-spi2-common.
(Reading database ... 122806 files and directories currently installed.)
Preparing to unpack .../0-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatk1.0-0t64:amd64.
Preparing to unpack .../1-libatk1.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatk1.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../2-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libatk-bridge2.0-0t64:amd64.
Preparing to unpack .../3-libatk-bridge2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatk-bridge2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxcomposite1:amd64.
Preparing to unpack .../4-libxcomposite1_1%3a0.4.5-1build3_amd64.deb ...
Unpacking libxcomposite1:amd64 (1:0.4.5-1bu

## 2. Importar librerías

In [4]:
import re, json, time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## 3. Elegir el artículo
Pega aquí el enlace del producto que escogiste en alkomprar.com (abre el producto y copia la dirección del navegador).

Antes de ejecutar, revisa https://www.alkomprar.com/robots.txt y los términos de uso. Este código hace **una sola consulta** y usa pausas.

In [5]:
URL_PRODUCTO = "https://www.alkomprar.com/celular-samsung-galaxy-a27-256gb-5g-negro/p/8806099361082"

## 4. Abrir el navegador y entrar a la página

In [6]:
opciones = Options()
opciones.add_argument("--headless=new")
opciones.add_argument("--no-sandbox")
opciones.add_argument("--disable-dev-shm-usage")
opciones.add_argument("--window-size=1366,900")
opciones.add_argument("--lang=es-CO")

driver = webdriver.Chrome(options=opciones)   # Selenium descarga el chromedriver solo
driver.get(URL_PRODUCTO)

WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
time.sleep(3)   # pausa para que cargue el precio y las opiniones
print("Página cargada:", driver.title)

Página cargada: Celular SAMSUNG Galaxy A27 256GB 5G Negro


## 5. Sacar los datos
Hay dos formas. La primera lee los datos estructurados (JSON-LD) que muchas tiendas incluyen en la página para Google. Si no están, se usan selectores CSS.

**Si algún dato sale vacío:** en tu navegador haz clic derecho sobre el precio → *Inspeccionar*, mira el nombre de la clase y cámbialo en el diccionario `SELECTORES`.

In [ ]:
# Selectores de respaldo: AJUSTA estos nombres mirando la página con "Inspeccionar"
def a_numero(texto):
    """Convierte '$1.089.050' en 1089050."""
    solo = re.sub(r"[^0-9]", "", texto or "")
    return int(solo) if solo else None

datos = {"nombre": None, "precio_actual": None, "precio_anterior": None,
         "calificacion": None, "opiniones": None, "url": URL_PRODUCTO}

# Nombre, calificación y opiniones (datos estructurados JSON-LD)
for script in driver.find_elements(By.CSS_SELECTOR, "script[type='application/ld+json']"):
    try:
        info = json.loads(script.get_attribute("innerHTML"))
    except Exception:
        continue
    for item in (info if isinstance(info, list) else [info]):
        if isinstance(item, dict) and item.get("@type") == "Product":
            datos["nombre"] = item.get("name")
            rating = item.get("aggregateRating", {})
            datos["calificacion"] = rating.get("ratingValue")
            datos["opiniones"] = rating.get("reviewCount")

if not datos["nombre"]:
    datos["nombre"] = driver.find_element(By.TAG_NAME, "h1").text.strip()

# Precios: se buscan por cómo se ven (el tachado es el anterior)
js = r"""
const out = [];
document.querySelectorAll('body *').forEach(el => {
  if (el.children.length > 0) return;
  const t = (el.textContent || '').trim();
  if (!/^\$\s?\d{1,3}(\.\d{3})+(\s*hoy)?$/i.test(t)) return;
  let tachado = false, p = el;
  while (p && p !== document.body) {
    if (['DEL','S','STRIKE'].includes(p.tagName) ||
        getComputedStyle(p).textDecorationLine.includes('line-through')) tachado = true;
    p = p.parentElement;
  }
  out.push({texto: t, tachado: tachado,
            tam: parseFloat(getComputedStyle(el).fontSize),
            visible: el.offsetParent !== null});
});
return out;
"""
precios = [p for p in driver.execute_script(js) if p["visible"]]
print("Precios encontrados:", precios)

tachados = [p for p in precios if p["tachado"]]
normales = [p for p in precios if not p["tachado"]]
if tachados:
    datos["precio_anterior"] = a_numero(tachados[0]["texto"])
# Porcentaje de descuento que muestra la página (ej. "-27%")
texto_pagina = driver.find_element(By.TAG_NAME, "body").text
m = re.search(r"-\s?(\d{1,2})\s?%", texto_pagina)
pct_pagina = int(m.group(1)) if m else None
print("Descuento que muestra la página:", pct_pagina)

valores = sorted({a_numero(p["texto"]) for p in normales if a_numero(p["texto"])})
anterior = datos["precio_anterior"]
if anterior:
    valores = [v for v in valores if v < anterior]

if valores and anterior and pct_pagina:
    # el precio actual es el que da el descuento más parecido al de la página
    datos["precio_actual"] = min(
        valores, key=lambda v: abs((anterior - v) / anterior * 100 - pct_pagina))
elif valores:
    datos["precio_actual"] = max(valores)

datos

Precios encontrados: [{'tachado': True, 'tam': 24, 'texto': '$1.499.050', 'visible': True}, {'tachado': False, 'tam': 18, 'texto': '$32.928', 'visible': True}, {'tachado': True, 'tam': 12, 'texto': '$1.499.050', 'visible': True}, {'tachado': False, 'tam': 16, 'texto': '$1.089.050', 'visible': True}, {'tachado': True, 'tam': 12, 'texto': '$49.900', 'visible': True}, {'tachado': False, 'tam': 16, 'texto': '$19.900', 'visible': True}, {'tachado': False, 'tam': 12, 'texto': '$440.000', 'visible': True}, {'tachado': False, 'tam': 12, 'texto': '$1.108.950', 'visible': True}, {'tachado': False, 'tam': 20, 'texto': '$32.928', 'visible': True}, {'tachado': True, 'tam': 14, 'texto': '$1.499.050', 'visible': True}, {'tachado': True, 'tam': 16, 'texto': '$1.499.050', 'visible': True}, {'tachado': False, 'tam': 20, 'texto': '$1.089.050', 'visible': True}, {'tachado': True, 'tam': 16, 'texto': '$6.999.990', 'visible': True}, {'tachado': False, 'tam': 20, 'texto': '$6.299.990', 'visible': True}, {'ta

{'nombre': 'Celular Samsung Galaxy A27 256gb 5G Negro',
 'precio_actual': 1089050,
 'precio_anterior': 1499050,
 'calificacion': '5.00',
 'opiniones': '13',
 'url': 'https://www.alkomprar.com/celular-samsung-galaxy-a27-256gb-5g-negro/p/8806099361082'}

## 6. Calcular el descuento y armar la tabla

In [2]:
precio = float(datos["precio_actual"]) if datos["precio_actual"] else None
anterior = datos["precio_anterior"]

if precio and anterior and anterior > precio:
    datos["descuento_%"] = round((anterior - precio) / anterior * 100, 1)
else:
    datos["descuento_%"] = 0   # sin descuento o dato no disponible

tabla = pd.DataFrame([datos])
tabla

NameError: name 'datos' is not defined

## 7. Guardar el resultado y cerrar el navegador

In [1]:
tabla.to_csv("alkomprar_articulo.csv", index=False, encoding="utf-8-sig")
driver.quit()
print("Archivo guardado: alkomprar_articulo.csv")

# Para descargarlo a tu computador:
from google.colab import files
files.download("alkomprar_articulo.csv")

NameError: name 'tabla' is not defined